# 🚀 Lightweight Linguistic Extraction (Single Tar Split)

Downloads a single `.tar` batch (`flac_T_aa.tar` - ~7GB, ~36,000 files) from Zenodo, transcribes them using `openai/whisper-small` on Colab T4 GPU, and splits them into Train, Dev, and Eval CSV datasets.

### Features:
1. **Full-Batch Transcription**: Processes the entire single tar file (~36,000 files) to give your linguistic model rich text data.
2. **Corrected ASVspoof 5 TSV Protocol Parser**: Parses the 10-column protocols correctly to avoid 1-class imbalance bugs.
3. **T4 GPU Pipeline Execution**: Uses Whisper FP16 with Hugging Face batching.

In [ ]:
# 1. Setup Environment
!pip install -q zenodo_get transformers accelerate datasets soundfile pandas numpy tqdm scikit-learn

In [ ]:
# 2. Mount Drive & Setup Paths
from google.colab import drive
import os, shutil, tarfile, json, requests, random
import pandas as pd, numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import torch
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/142_Feature_Extracted')
LING_DIR = BASE_DIR / 'Linguistic' / 'Dataset'
LING_DIR.mkdir(parents=True, exist_ok=True)

TEMP_DIR = Path('/content/temp_workspace_gpu')
TEMP_FLACS = TEMP_DIR / 'flacs'
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("Workspace Ready.")

In [ ]:
# 3. Fetch Zenodo Info & Load Corrected Protocols
ZENODO_RECORD = '14498691'
API_URL = f"https://zenodo.org/api/records/{ZENODO_RECORD}"
res = requests.get(API_URL).json()
files_info = {f['key']: f['links']['self'] for f in res['files']}

PROTOCOL_TAR = 'ASVspoof5_protocols.tar.gz'
PROTOCOL_DIR = BASE_DIR / 'protocols'

if not PROTOCOL_DIR.exists():
    print("Downloading protocols from Zenodo...")
    PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
    !zenodo_get -r {ZENODO_RECORD} -g {PROTOCOL_TAR} -o {TEMP_DIR}
    with tarfile.open(TEMP_DIR / PROTOCOL_TAR, 'r:gz') as tar: 
        tar.extractall(path=PROTOCOL_DIR)
    (TEMP_DIR / PROTOCOL_TAR).unlink()

protocol_map = {}
protocol_files = list(PROTOCOL_DIR.rglob('*.tsv')) + list(PROTOCOL_DIR.rglob('*.txt'))
print(f"Found {len(protocol_files)} protocol files. Parsing labels...")

for split_file in protocol_files:
    with open(split_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('#') or line.startswith('speaker_id') or not line.strip(): 
                continue
            line_lower = line.lower()
            if 'bonafide' in line_lower:
                label = 0
            elif 'spoof' in line_lower:
                label = 1
            else:
                continue
            parts = line.strip().split()
            fname = parts[1] if parts[1].endswith('.flac') else parts[1] + '.flac'
            protocol_map[fname] = label

print(f"Successfully loaded {len(protocol_map):,} entries into protocol_map.")

In [ ]:
# 4. Initialize Whisper-Small on GPU
from transformers import pipeline
device = 0 if torch.cuda.is_available() else -1
print(f"Initializing Whisper on device {device}...")

transcriber = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-small',
    device=device,
    generate_kwargs={'language': 'english', 'task': 'transcribe'},
    chunk_length_s=30,
    torch_dtype=torch.float16 if device == 0 else torch.float32,
)

In [ ]:
# 5. Whisper Batch Transcriber Function
def transcribe_batch(file_paths):
    from datasets import Dataset, Audio
    valid_paths = [f for f in file_paths if f.name in protocol_map]
    if not valid_paths:
        print("⚠️ No valid audio paths found in protocol_map!")
        return []
    
    ds = Dataset.from_dict({'audio': [str(p) for p in valid_paths], 'filename': [p.name for p in valid_paths]})
    ds = ds.cast_column('audio', Audio(sampling_rate=16000))
    
    def audio_gen(dataset):
        for sample in dataset: yield sample['audio']
        
    results = []
    for sample, out in zip(ds, transcriber(audio_gen(ds), batch_size=16)):
        results.append({
            'filename': sample['filename'],
            'text': (out.get('text') or '').strip(),
            'label': protocol_map[sample['filename']]
        })
    return results

In [ ]:
# 6. Single Tar Download & Extraction
SINGLE_TAR = 'flac_T_aa.tar'

print(f"\n{'='*50}\nProcessing {SINGLE_TAR}...\n{'='*50}")
url = files_info[SINGLE_TAR]
tar_path = TEMP_DIR / SINGLE_TAR

print("Downloading...")
!wget -q -O {tar_path} {url}

if TEMP_FLACS.exists(): shutil.rmtree(TEMP_FLACS)
TEMP_FLACS.mkdir(parents=True, exist_ok=True)

print("Extracting audio files...")
with tarfile.open(tar_path, 'r') as tar: 
    if hasattr(tarfile, 'data_filter'):
        tar.extractall(path=TEMP_FLACS, filter='data')
    else:
        tar.extractall(path=TEMP_FLACS)

all_flacs = list(TEMP_FLACS.rglob('*.flac'))
print(f"Found {len(all_flacs):,} audio files in tar. Ready for full transcription.")

In [ ]:
# 7. Transcription & Automatic Splitting
print(f"Transcribing via T4 GPU...")
t_results = transcribe_batch(all_flacs)

if t_results:
    print("\nTranscription complete! Splitting data into Train/Dev/Eval (80/10/10)...")
    df = pd.DataFrame(t_results)
    
    # Split 80% Train, 20% Temp
    train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
    
    # Split Temp into 50% Dev, 50% Eval
    dev_df, eval_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
    
    train_csv = LING_DIR / "transcripts_train.csv"
    dev_csv = LING_DIR / "transcripts_dev.csv"
    eval_csv = LING_DIR / "transcripts_eval.csv"
    
    train_df.to_csv(train_csv, index=False)
    dev_df.to_csv(dev_csv, index=False)
    eval_df.to_csv(eval_csv, index=False)
    
    print(f"✅ Saved Train: {len(train_df):,} rows")
    print(f"✅ Saved Dev: {len(dev_df):,} rows")
    print(f"✅ Saved Eval: {len(eval_df):,} rows")
else:
    print("⚠️ No features were extracted.")

print("Cleaning up temporary files...")
shutil.rmtree(TEMP_FLACS)
tar_path.unlink()
print("\n🎉 Extraction and Splitting Complete!")